## Import du dataframe

In [ ]:
import pandas as pd

df_conjugality = pd.read_csv('../data/src/conjugality.csv', sep=';', dtype={'GEO': str})
df_result = pd.read_csv('../data/src/results.csv', low_memory=False)
df_elec = pd.read_csv('../data/src/election.csv', sep=';', dtype={'Code_INSEE': str})

display(df_elec)


## Attempt to fuse them

In [ ]:
df_fuse = df_conjugality.merge(
    df_elec,
    left_on="GEO",
    right_on="Code_INSEE",
    how="inner"
)

display(df_fuse.head())

age_mapping = {
    '15 ans ou plus': 0,
    'De 15 à 24 ans': 1,
    'De 25 à 39 ans': 2,
    'De 40 à 54 ans': 3,
    'De 55 à 64 ans': 4,
    'De 65 à 79 ans': 5,
    '80 ans ou plus': 6
}
df_fuse['AGE'] = df_fuse['AGE'].map(age_mapping)

civil_status_mapping = {
    'Célibataire': 0,
    'Marié': 1,
    'Pacsé': 2,
    'En concubinage, union libre': 3,
    'Divorcé': 4,
    'Total': 5
}
df_fuse['CIVIL_STATUS'] = df_fuse['CIVIL_STATUS'].map(civil_status_mapping)

couple_mapping = {
    'Non': 0,
    'Oui': 1,
    'Total': 2
}
df_fuse['COUPLE'] = df_fuse['COUPLE'].map(couple_mapping)

## Trying to find correlation

In [ ]:
import seaborn as sns

import matplotlib.pyplot as plt
age_cols = [col for col in df_fuse.columns if 'age' in col.lower()]
pct_cols = ['pct_blancs', 'pct_nuls', 'pct_gauche', 'pct_droite', 'pct_centre']
selected_cols = age_cols + pct_cols

corr_selected = df_fuse[selected_cols].corr()

sns.heatmap(corr_selected, annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap for Selected Columns')
plt.show()

def corr_and_heatmap(one_hot, title_base):
    corr_df = pd.concat([df_fuse[pct_cols], one_hot], axis=1).corr()
    corr_part = corr_df.loc[pct_cols, one_hot.columns]
    print(f"Correlation between pct_* variables and {title_base} categories:")
    print(corr_part)
    sns.heatmap(corr_part, annot=True, cmap='coolwarm')
    plt.title(f'Correlation pct_* vs {title_base} category (one-hot)')
    plt.xlabel(f'{title_base} category')
    plt.ylabel('pct variables')
    plt.show()

age_dummies = pd.get_dummies(df_fuse['AGE'], prefix='AGE')
corr_age_pct = pd.concat([df_fuse[pct_cols], age_dummies], axis=1).corr().loc[pct_cols, age_dummies.columns]
print("Correlation between pct_* variables and AGE categories:")
print(corr_age_pct)
sns.heatmap(corr_age_pct, annot=True, cmap='coolwarm')
plt.title('Correlation pct_* vs AGE category (ages as columns)')
plt.xlabel('AGE category')
plt.ylabel('pct variables')
plt.show()

civil_dummies = pd.get_dummies(df_fuse['CIVIL_STATUS'], prefix='')
corr_and_heatmap(civil_dummies, 'CIVIL_STATUS')

couple_dummies = pd.get_dummies(df_fuse['COUPLE'], prefix='COUPLE')
corr_and_heatmap(couple_dummies, 'COUPLE')